# HW4

Name: Jeevika Mekala<br>
Github Username: Jeevikaa<br>
USC ID: 4369563404

## 1. Time Series Classification Part 1: Feature Creation/Extraction (HW3 Rerun)
### (a) Obtain Data

Import packages

In [1]:
import os, re, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import bootstrap
from sklearn.preprocessing import StandardScaler, MinMaxScaler, label_binarize
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc,
                             roc_auc_score, accuracy_score)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

Get the AReM Data Set

In [2]:
FILEPATH = r"C:\Users\jeevi\Downloads\USC\SEM-2\DSML\HW4\Homework 3, 4 Data\AReM"
L_MAX = 20
SERIES = ["avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
SERIES_ID = {s: i + 1 for i, s in enumerate(SERIES)}
STATS = {
    "min": np.min, "max": np.max, "mean": np.mean, "median": np.median,
    "std": lambda v: np.std(v, ddof=1),
    "q1": lambda v: np.percentile(v, 25), "q3": lambda v: np.percentile(v, 75),
}
SEED = 69

In [3]:
def read_instance(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            rows.append([float(x) for x in re.split(r"[,\s]+", line) if x])
    return pd.DataFrame(np.array(rows)[:, 1:], columns=SERIES)

def fix_dropouts(df):
    df, n = df.copy(), 0
    for avg, var in [("avg_rss12", "var_rss12"), ("avg_rss13", "var_rss13"), ("avg_rss23", "var_rss23")]:
        a = df[avg].values
        for i in range(1, len(a) - 1):
            if a[i] == 0 and a[i - 1] > 5 and a[i + 1] > 5:
                df.loc[i, [avg, var]] = np.nan
                n += 1
    return df.interpolate(limit_direction="both"), n

instances, repaired = [], {}
for folder in sorted(os.listdir(FILEPATH)):
    path = os.path.join(FILEPATH, folder)
    if not os.path.isdir(path):
        continue
    for fname in sorted(os.listdir(path)):
        m = re.fullmatch(r"dataset(\d+)\.csv", fname)
        if not m:
            continue
        df, n = fix_dropouts(read_instance(os.path.join(path, fname)))
        if len(df) != 480:
            print(f"note: {folder}/{fname} has {len(df)} rows (expected 480), keeping it")
        if n:
            repaired[f"{folder}/{fname}"] = n
        instances.append({"activity": folder, "idx": int(m.group(1)), "df": df})

print(len(instances), "instances")
print(pd.Series([i["activity"] for i in instances]).value_counts().sort_index().to_dict())
print("sensor dropouts repaired:", repaired)

note: sitting/dataset8.csv has 479 rows (expected 480), keeping it
88 instances
{'bending1': 7, 'bending2': 6, 'cycling': 15, 'lying': 15, 'sitting': 15, 'standing': 15, 'walking': 15}
sensor dropouts repaired: {'bending2/dataset2.csv': 3, 'cycling/dataset9.csv': 2, 'lying/dataset1.csv': 1, 'lying/dataset10.csv': 1, 'lying/dataset15.csv': 1, 'lying/dataset6.csv': 2, 'sitting/dataset15.csv': 1, 'sitting/dataset2.csv': 1, 'standing/dataset1.csv': 1, 'standing/dataset12.csv': 2, 'standing/dataset13.csv': 1}


### (b) Splitting Data

In [4]:
def is_test(inst):
    return inst["idx"] in ((1, 2) if inst["activity"].startswith("bending") else (1, 2, 3))

train = [i for i in instances if not is_test(i)]
test = [i for i in instances if is_test(i)]
print("train:", len(train), "| test:", len(test))
print("train per activity:", pd.Series([i["activity"] for i in train]).value_counts().sort_index().to_dict())

train: 69 | test: 19
train per activity: {'bending1': 5, 'bending2': 4, 'cycling': 12, 'lying': 12, 'sitting': 12, 'standing': 12, 'walking': 12}


### (c) Feature Extraction
#### i. Common time-domain features
* Minimum and maximum: range of the signal
* Mean and median: central level of the signal
* Standard deviation: spread around the mean
* First and third quartile (25th and 75th percentile): shape of the distribution
* Interquartile range: Q3 - Q1, a spread measure that is not affected much by outliers
* Skewness and kurtosis: asymmetry and tail weight
* Root mean square / energy: overall magnitude of the signal
* Zero-crossing (mean-crossing) rate: how often the series crosses its mean
* Auto-correlation: similarity of the series with a lagged copy of itself
* Cross-correlation: similarity between two different series

#### ii. Extract features
Min, max, mean, median, standard deviation, first and third quartile for each of the 6 series.

In [5]:
def short(stat, series, seg=None, l=1):
    return f"{stat}{SERIES_ID[series]}" + (f"_seg{seg}" if l > 1 else "")

def feature_row(df, l, series, stats):
    row = {}
    for seg, part in enumerate(np.array_split(df[series].values, l, axis=0), start=1):
        for si, s in enumerate(series):
            for st in stats:
                row[short(st, s, seg, l)] = STATS[st](part[:, si])
    return row

def feature_table(insts, l=1, series=SERIES, stats=list(STATS)):
    return pd.DataFrame([feature_row(i["df"], l, series, stats) for i in insts])

df_features = feature_table(instances, l=1)
df_features.insert(0, "Instance", [f"{i['activity']}/dataset{i['idx']}" for i in instances])
df_features

,Instance,min1,max1,mean1,median1,std1,q11,q31,min2,max2,...,std5,q15,q35,min6,max6,mean6,median6,std6,q16,q36
0,bending1/dataset1,37.25,45.00,40.624792,40.500,1.476967,39.25,42.0000,0.0,1.30,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.430,0.582915,0.00,1.3000
1,bending1/dataset2,38.00,45.67,42.812812,42.500,1.435550,42.00,43.6700,0.0,1.22,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.430,0.601010,0.00,1.3000
2,bending1/dataset3,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,0.0,1.70,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.430,0.513506,0.00,0.9400
3,bending1/dataset4,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,0.0,3.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.500,0.524317,0.00,1.0000
4,bending1/dataset5,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,0.0,2.83,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.430,0.389164,0.00,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,walking/dataset5,20.75,46.25,34.763333,35.290,4.742208,31.67,38.2500,0.0,12.68,...,3.174681,14.2500,18.33,0.0,9.39,3.288271,3.270,1.647528,2.05,4.3050
84,walking/dataset6,21.50,51.00,34.935812,35.500,4.645944,32.00,38.0625,0.0,12.21,...,3.192058,14.2375,18.25,0.0,10.21,3.280021,3.015,1.700918,2.12,4.5000
85,walking/dataset7,18.33,47.67,34.333042,34.750,4.948770,31.25,38.0000,0.0,12.48,...,3.000493,13.7500,18.00,0.0,8.01,3.261583,2.980,1.617290,2.05,4.3200
86,walking/dataset8,18.33,45.75,34.599875,35.125,4.731790,31.50,38.0000,0.0,15.37,...,2.905688,14.0000,18.25,0.0,8.86,3.289542,3.015,1.680170,2.12,4.2600


#### iii. Standard Deviation and bootstrap confidence intervals

In [6]:
feat_only = df_features.drop(columns="Instance")
feat_only.std(ddof=1).rename("std").to_frame()

,std
min1,8.668533
max1,4.394362
mean1,5.334590
median1,5.440054
std1,1.792073
q11,6.153590
q31,5.138925
min2,0.000000
max2,5.062729
mean2,1.574153


In [ ]:
rows = []
for col in feat_only.columns:
    x = feat_only[col].values
    if np.ptp(x) == 0:
        rows.append((col, 0.0, 0.0, 0.0)); continue
    res = bootstrap((x,), lambda v, axis: np.std(v, axis=axis, ddof=1), confidence_level=0.90,
                    n_resamples=5000, method="percentile", random_state=1)
    rows.append((col, np.std(x, ddof=1), res.confidence_interval.low, res.confidence_interval.high))
ci = pd.DataFrame(rows, columns=["Feature", "Std", "Lower Bound (90%)", "Upper Bound (90%)"])
ci["Relative width"] = (ci["Upper Bound (90%)"] - ci["Lower Bound (90%)"]) / ci["Std"].replace(0, np.nan)
ci

#### iv. Select features
Selected features: mean, standard deviation and maximum.
1. Mean: has one of the narrower bootstrap intervals above, so it is a stable feature.
2. Standard deviation: moving activities (bending, walking, cycling) vary more than still ones (lying, sitting, standing).
3. Maximum: picks up the spikes in signal strength during a movement.

The minimum of the var series is 0 for every instance (std 0 in the table), so it is not used.

In [ ]:
SELECTED_STATS = ["mean", "std", "max"]
SCATTER_SERIES = ["avg_rss12", "var_rss12", "var_rss23"]

## 2. Time Series Classification Part 2: Binary and Multiclass Classification
### (a) Binary Classification Using Logistic Regression
#### i. Plots

In [ ]:
def label_of(inst, binary=True):
    return int("bending" in inst["activity"]) if binary else inst["activity"]

def make_xy(insts, l=1, series=SERIES, stats=SELECTED_STATS, binary=True):
    X = feature_table(insts, l, series, stats)
    y = pd.Series([label_of(i, binary) for i in insts], name="type")
    return X, y

X, y = make_xy(train, 1, SCATTER_SERIES)
sns.pairplot(pd.concat([X, y], axis=1), hue="type", palette={1: "red", 0: "blue"})
plt.show()

#### ii. Split each series into two parts

In [ ]:
X, y = make_xy(train, 2, SCATTER_SERIES)
sns.pairplot(pd.concat([X, y], axis=1), hue="type", palette={1: "red", 0: "blue"})
plt.show()

#### Observation:
With l = 1 the red points (bending) mostly sit in the same area as a lot of the blue ones. The panels with mean2, std2, mean6 and std6 split off a group of blue points (the high motion activities), but the red points are not cleanly separated in any single panel. With l = 2 the plots look almost the same as before, so splitting the series in two did not really help me separate the classes by eye. Bending is also a small group (only 9 training instances), and it falls in the low variance region next to lying, sitting and standing.

#### iii. Time Series Segments

In [ ]:
def standardize(Xtr):
    sc = StandardScaler().fit(Xtr)
    return sc, sc.transform(Xtr)

def rfecv_for_l(insts, l, seed=SEED):
    X, y = make_xy(insts, l)
    sc, Z = standardize(X)
    est = LogisticRegression(C=1e8, max_iter=5000)
    sel = RFECV(est, step=max(1, Z.shape[1] // 20), min_features_to_select=1, scoring="accuracy",
                cv=StratifiedKFold(5, shuffle=True, random_state=seed))
    sel.fit(Z, y)
    return sc, sel, list(X.columns)

fits, rows = {}, []
for l in range(1, L_MAX + 1):
    sc, sel, names = rfecv_for_l(train, l)
    fits[l] = (sc, sel, names)
    rows.append({"l": l, "p": sel.n_features_, "CV accuracy": sel.cv_results_["mean_test_score"].max(),
                 "CV std": sel.cv_results_["std_test_score"][np.argmax(sel.cv_results_["mean_test_score"])]})
cv_table = pd.DataFrame(rows)
cv_table

In [ ]:
best = cv_table.loc[cv_table["CV accuracy"].idxmax()]
l_best, p_best = int(best["l"]), int(best["p"])
print(f"Best (l, p) = ({l_best}, {p_best}) with CV accuracy {best['CV accuracy']:.4f}")
sc_best, sel_best, names_best = fits[l_best]
rfe_features = [n for n, keep in zip(names_best, sel_best.support_) if keep]
print("RFE selected features:", rfe_features)

#### The right way vs the wrong way to cross-validate
Wrong way: select the features on the whole training set first and then cross-validate only the final fit. The validation folds have already influenced which features were kept, so the CV accuracy is too optimistic.

Right way (used here): split into folds first, and do the feature elimination inside each fold using only that fold's training part. l and p are chosen from those scores.
Because there are only 9 bending instances, I used stratified 5-fold CV so every fold has some of them.

#### iv. Confusion Matrix

In [ ]:
def fit_final(insts, l, support, names):
    X, y = make_xy(insts, l)
    sc, Z = standardize(X)
    lr = LogisticRegression(C=1e8, max_iter=5000).fit(Z[:, support], y)
    return sc, lr

def predict_final(sc, support, lr, insts, l):
    X, y = make_xy(insts, l)
    Z = sc.transform(X)[:, support]
    return y.values, lr.predict(Z), lr.predict_proba(Z)[:, 1]

def report_binary(y_true, pred, prob, title):
    cm = confusion_matrix(y_true, pred)
    fpr, tpr, _ = roc_curve(y_true, prob)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ConfusionMatrixDisplay(cm, display_labels=["other", "bending"]).plot(ax=ax[0], colorbar=False)
    ax[0].set_title(f"{title}: confusion matrix")
    ax[1].plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.4f}")
    ax[1].plot([0, 1], [0, 1], "--"); ax[1].set_xlabel("False Positive Rate"); ax[1].set_ylabel("True Positive Rate")
    ax[1].set_title(f"{title}: ROC"); ax[1].legend()
    plt.show()
    print(f"{title}: accuracy = {accuracy_score(y_true, pred):.4f}, AUC = {auc(fpr, tpr):.4f}")

support = sel_best.support_
sc_lr, lr_best = fit_final(train, l_best, support, names_best)
y_tr, pred_tr, prob_tr = predict_final(sc_lr, support, lr_best, train, l_best)
report_binary(y_tr, pred_tr, prob_tr, "Train")

In [ ]:
Xtr_l, ytr_l = make_xy(train, l_best)
Ztr = sc_lr.transform(Xtr_l)[:, support]
betas = pd.DataFrame({"Feature": ["intercept"] + rfe_features, "beta (standardised features)": np.r_[lr_best.intercept_, lr_best.coef_[0]]})
try:
    logit = sm.Logit(ytr_l.values.astype(float), sm.add_constant(Ztr)).fit(method="bfgs", maxiter=1000, disp=0)
    betas["p-value"] = logit.pvalues
except Exception as e:
    betas["p-value"] = np.nan
    print("statsmodels could not compute p-values:", type(e).__name__)
betas

#### v. Test the classifier

In [ ]:
y_te, pred_te, prob_te = predict_final(sc_lr, support, lr_best, test, l_best)
report_binary(y_te, pred_te, prob_te, "Test")
print(f"Cross-validation accuracy: {best['CV accuracy']:.4f} | test accuracy: {accuracy_score(y_te, pred_te):.4f}")

#### Observation:
The best (l, p) from CV was (4, 1) with a CV accuracy of 0.971, and the one feature it kept was max6_seg3. On the test set the accuracy is only 0.789 and the AUC is 0.617. The test set has 4 bending and 15 other instances, so 0.789 is exactly what I would get by predicting "other" every time (15/19). So the model is not really picking up bending on new data, and the CV accuracy was too optimistic. With 9 bending training instances and a single feature that is not too surprising.

#### vi. Separation

In [ ]:
train_acc = accuracy_score(y_tr, pred_tr)
max_beta = np.abs(lr_best.coef_).max()
print(f"train accuracy = {train_acc:.4f}, largest |beta| on standardised features = {max_beta:.2f}")
print("p-values:", betas["p-value"].round(3).tolist())
print("=> classes look (nearly) perfectly separable" if train_acc == 1 and (max_beta > 10 or betas["p-value"].isna().any() or (betas["p-value"] > 0.9).any())
      else "=> no sign of complete separation")

Answer:

No, the classes are not perfectly separated here. The training accuracy is 0.87, the betas are small (largest about 1.47 on standardized features) and the p-values are small (0.00004 for the intercept and 0.018 for max6_seg3). If the classes were perfectly separable the likelihood would keep increasing, the betas would blow up and the p-values would be close to 1, which is not what happens here.

#### vii. Imbalance

In [ ]:
counts = pd.Series(y_tr).value_counts().sort_index()
print("class counts in training data (0 = other, 1 = bending):", counts.to_dict())
print(f"bending fraction = {counts[1] / counts.sum():.3f}")

Only 13% of the training data is bending (9 out of 69), so the classes are imbalanced. I used case-control sampling: keep all bending instances, draw the same number of non-bending ones at random, fit, and adjust the intercept so the predicted probabilities match the original class proportions.

Observation: the training accuracy and AUC stay the same (0.870 and 0.804) and the test AUC is still 0.617. The test accuracy goes down from 0.789 to 0.737 because the model now predicts some instances as bending and gets a few wrong. Balancing the sample does not fix the weak signal in the single feature.

In [ ]:
rng = np.random.default_rng(SEED)
pos = np.where(ytr_l.values == 1)[0]
neg = rng.choice(np.where(ytr_l.values == 0)[0], size=len(pos), replace=False)
idx = np.r_[pos, neg]
Zfull = sc_lr.transform(Xtr_l)[:, support]
lr_cc = LogisticRegression(C=1e8, max_iter=5000).fit(Zfull[idx], ytr_l.values[idx])
pi, pi_s = ytr_l.mean(), ytr_l.values[idx].mean()
lr_cc.intercept_ = lr_cc.intercept_ + np.log(pi / (1 - pi)) - np.log(pi_s / (1 - pi_s))

for name, insts in [("Train (case-control model)", train), ("Test (case-control model)", test)]:
    yy, pp, pr = predict_final(sc_lr, support, lr_cc, insts, l_best)
    report_binary(yy, pp, pr, name)

### (b) Binary Classification Using L1-penalized logistic regression
#### i. Time Series Segments

In [ ]:
Cs = np.logspace(-2, 2, 20)
def l1_cv(insts, l, binary=True):
    X, y = make_xy(insts, l, binary=binary)
    sc, Z = standardize(X)
    m = LogisticRegressionCV(Cs=Cs, penalty="l1", solver="liblinear", scoring="accuracy",
                             cv=StratifiedKFold(5, shuffle=True, random_state=SEED), max_iter=2000).fit(Z, y)
    return sc, m

rows, l1_models = [], {}
for l in range(1, L_MAX + 1):
    sc, m = l1_cv(train, l)
    l1_models[l] = (sc, m)
    cv_acc = m.scores_[1].mean(axis=0).max()
    rows.append({"l": l, "best C": m.C_[0], "non-zero coefficients": int((m.coef_ != 0).sum()), "CV accuracy": cv_acc})
l1_table = pd.DataFrame(rows)
l1_table

In [ ]:
l1_best = int(l1_table.loc[l1_table["CV accuracy"].idxmax(), "l"])
sc1, m1 = l1_models[l1_best]
print(f"Best l = {l1_best}, C = {m1.C_[0]:.4g}, CV accuracy = {l1_table['CV accuracy'].max():.4f}")
for name, insts in [("Train (L1)", train), ("Test (L1)", test)]:
    X_, y_ = make_xy(insts, l1_best)
    Z_ = sc1.transform(X_)
    report_binary(y_.values, m1.predict(Z_), m1.predict_proba(Z_)[:, 1], name)

#### ii. Comparison
The L1 model (best l = 20, C = 1.27) has a CV accuracy of 0.986, training accuracy 1.0, test accuracy 0.947 and test AUC 0.867. The RFE model from (a) only got 0.789 and 0.617 on the test set, so L1 is clearly better here. L1 selects features and fits the model in one step and keeps several features, while RFE ended up with just one. Training accuracy of 1.0 with l = 20 means it fits the training data perfectly, so it may overfit a bit, but the test accuracy is still good. L1 is also easier to implement: LogisticRegressionCV does the cross-validation for C and the selection in one call, while the RFE approach needs the elimination loop plus statsmodels for the p-values.

In [ ]:
CLASSES = sorted({i["activity"] for i in instances})
def eval_multiclass(model, Z, y, title):
    pred = model.predict(Z)
    prob = model.predict_proba(Z)
    cm = confusion_matrix(y, pred, labels=CLASSES)
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax[0], colorbar=False, xticks_rotation=45)
    ax[0].set_title(f"{title}: confusion matrix")
    Yb = label_binarize(y, classes=CLASSES)
    aucs = {}
    for k, c in enumerate(CLASSES):
        if 0 < Yb[:, k].sum() < len(Yb):
            fpr, tpr, _ = roc_curve(Yb[:, k], prob[:, list(model.classes_).index(c)])
            aucs[c] = auc(fpr, tpr)
            ax[1].plot(fpr, tpr, label=f"{c} (AUC {aucs[c]:.3f})")
    ax[1].plot([0, 1], [0, 1], "--"); ax[1].set_title(f"{title}: one-vs-rest ROC"); ax[1].legend(fontsize=8)
    ax[1].set_xlabel("False Positive Rate"); ax[1].set_ylabel("True Positive Rate")
    plt.show()
    err = 1 - accuracy_score(y, pred)
    print(f"{title}: error = {err:.4f}, macro one-vs-rest AUC = {np.mean(list(aucs.values())):.4f}")
    return err, np.mean(list(aucs.values()))

rows, mn_models = [], {}
for l in range(1, L_MAX + 1):
    X_, y_ = make_xy(train, l, binary=False)
    sc, Z_ = standardize(X_)
    m = LogisticRegressionCV(Cs=np.logspace(-2, 2, 8), penalty="l1", solver="saga", scoring="accuracy",
                             cv=StratifiedKFold(5, shuffle=True, random_state=SEED), max_iter=2000, tol=1e-3).fit(Z_, y_)
    mn_models[l] = (sc, m)
    cv_acc = np.mean([s.mean(axis=0).max() for s in [np.mean(np.stack(list(m.scores_.values())), axis=0)]])
    rows.append({"l": l, "best C": m.C_[0], "CV accuracy": cv_acc})
mn_table = pd.DataFrame(rows)
mn_table

In [ ]:
mn_best = int(mn_table.loc[mn_table["CV accuracy"].idxmax(), "l"])
sc_mn, m_mn = mn_models[mn_best]
print(f"Best l = {mn_best}, CV accuracy = {mn_table['CV accuracy'].max():.4f}")
results = {}
Xte_, yte_ = make_xy(test, mn_best, binary=False)
results["L1 multinomial"] = eval_multiclass(m_mn, sc_mn.transform(Xte_), yte_, "Test (L1 multinomial)")

#### ii. Naive Bayes

In [ ]:
def nb_search(kind):
    rows = []
    for l in range(1, L_MAX + 1):
        X_, y_ = make_xy(train, l, binary=False)
        if kind == "gaussian":
            Z_, model = X_.values, GaussianNB()
        else:
            Z_, model = MinMaxScaler().fit_transform(X_), MultinomialNB()
        cv = cross_val_score(model, Z_, y_, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="accuracy")
        rows.append({"l": l, "CV accuracy": cv.mean()})
    return pd.DataFrame(rows)

def nb_fit_eval(kind, l, title):
    X_, y_ = make_xy(train, l, binary=False)
    Xt_, yt_ = make_xy(test, l, binary=False)
    if kind == "gaussian":
        model, Ztr, Zte = GaussianNB(), X_.values, Xt_.values
    else:
        mm = MinMaxScaler().fit(X_)
        model, Ztr, Zte = MultinomialNB(), mm.transform(X_), np.clip(mm.transform(Xt_), 0, 1)
    model.fit(Ztr, y_)
    return eval_multiclass(model, Zte, yt_, title)

gnb_table = nb_search("gaussian")
gnb_table

In [ ]:
l_g = int(gnb_table.loc[gnb_table["CV accuracy"].idxmax(), "l"])
print(f"Gaussian NB: best l = {l_g}")
results["Gaussian NB"] = nb_fit_eval("gaussian", l_g, "Test (Gaussian NB)")

In [ ]:
mnb_table = nb_search("multinomial")
mnb_table

In [ ]:
l_m = int(mnb_table.loc[mnb_table["CV accuracy"].idxmax(), "l"])
print(f"Multinomial NB: best l = {l_m}")
results["Multinomial NB"] = nb_fit_eval("multinomial", l_m, "Test (Multinomial NB)")

#### ii. Comparison (Gaussian vs Multinomial prior)

In [ ]:
pd.DataFrame(results, index=["Test error", "Macro AUC"]).T

Observation:
Gaussian NB gets a test error of 0.105 and macro AUC of 1.000, and Multinomial NB gets 0.211 and 0.966, so Gaussian NB is better. That makes sense because our features (means, std, max) are continuous, while Multinomial NB is meant for count data. The best l was 1 for Gaussian NB (CV accuracy 0.841) and 7 for Multinomial NB (CV accuracy 0.737).

#### iii. Best result
L1 multinomial logistic regression (best l = 1, CV accuracy 0.870) has a test error of 0.105 and macro AUC of 0.997. Gaussian NB has the same error (0.105) with AUC 1.000, and Multinomial NB is the worst. The test set only has 19 instances, so one mistake changes the error by about 0.05 and the first two are basically tied. I would go with Gaussian NB since it is simpler and has the higher AUC, but I would not read too much into the difference.

## 3. ISLR 4.8.3

With $K$ classes and a one-dimensional $X$ that is normal within each class with class-specific variance, the posterior is

$$p_k(x)=\frac{\pi_k\frac{1}{\sqrt{2\pi}\sigma_k}\exp\left(-\frac{(x-\mu_k)^2}{2\sigma_k^2}\right)}{\sum_{l=1}^{K}\pi_l\frac{1}{\sqrt{2\pi}\sigma_l}\exp\left(-\frac{(x-\mu_l)^2}{2\sigma_l^2}\right)}$$

The Bayes classifier picks the class with the largest $p_k(x)$. The denominator does not depend on $k$ and $\log$ is increasing, so it is enough to maximise

$$\delta_k(x)=\log\pi_k-\log\sigma_k-\frac{(x-\mu_k)^2}{2\sigma_k^2}
= -\frac{1}{2\sigma_k^2}x^2+\frac{\mu_k}{\sigma_k^2}x-\frac{\mu_k^2}{2\sigma_k^2}+\log\pi_k-\log\sigma_k .$$

The coefficient of $x^2$ is $-\frac{1}{2\sigma_k^2}$, which depends on $k$ whenever the variances differ. Hence $\delta_k(x)$ is a **quadratic** function of $x$ and the Bayes decision boundary is quadratic.

## 4. ISLR 4.8.7

By Bayes' theorem with $\pi_1=0.8$ (pays a dividend, mean $10$), $\pi_2=0.2$ (does not, mean $0$) and common variance $\sigma^2=36$:

$$P(Y=\text{Yes}\mid X=4)=\frac{\pi_1 e^{-\frac{(4-10)^2}{2\cdot 36}}}{\pi_1 e^{-\frac{(4-10)^2}{2\cdot 36}}+\pi_2 e^{-\frac{(4-0)^2}{2\cdot 36}}}
=\frac{0.8\,e^{-0.5}}{0.8\,e^{-0.5}+0.2\,e^{-0.2222}}$$

The common factor $\frac{1}{\sqrt{2\pi}\sigma}$ cancels, which gives about 0.752. So the probability that the company issues a dividend this year is about 0.752.